In [3]:
import cv2
import os

label = "helmet"  # change to "no_helmet"

save_dir = f"dataset/{label}"
os.makedirs(save_dir, exist_ok=True)

cap = cv2.VideoCapture(1)
count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Capture", frame)

    key = cv2.waitKey(1)

    if key == ord('s'):
        filename = f"{save_dir}/img_{count}.jpg"
        cv2.imwrite(filename, frame)
        print("Saved:", filename)
        count += 1

    elif key == 27:
        break

cap.release()
cv2.destroyAllWindows()

Saved: dataset/helmet/img_0.jpg
Saved: dataset/helmet/img_1.jpg
Saved: dataset/helmet/img_2.jpg
Saved: dataset/helmet/img_3.jpg
Saved: dataset/helmet/img_4.jpg
Saved: dataset/helmet/img_5.jpg
Saved: dataset/helmet/img_6.jpg
Saved: dataset/helmet/img_7.jpg
Saved: dataset/helmet/img_8.jpg
Saved: dataset/helmet/img_9.jpg
Saved: dataset/helmet/img_10.jpg
Saved: dataset/helmet/img_11.jpg
Saved: dataset/helmet/img_12.jpg
Saved: dataset/helmet/img_13.jpg
Saved: dataset/helmet/img_14.jpg
Saved: dataset/helmet/img_15.jpg
Saved: dataset/helmet/img_16.jpg
Saved: dataset/helmet/img_17.jpg
Saved: dataset/helmet/img_18.jpg
Saved: dataset/helmet/img_19.jpg
Saved: dataset/helmet/img_20.jpg
Saved: dataset/helmet/img_21.jpg
Saved: dataset/helmet/img_22.jpg
Saved: dataset/helmet/img_23.jpg
Saved: dataset/helmet/img_24.jpg
Saved: dataset/helmet/img_25.jpg
Saved: dataset/helmet/img_26.jpg
Saved: dataset/helmet/img_27.jpg
Saved: dataset/helmet/img_28.jpg
Saved: dataset/helmet/img_29.jpg
Saved: dataset/helme

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 224
batch_size = 16

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    "dataset",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val = datagen.flow_from_directory(
    "dataset",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(train, validation_data=val, epochs=5)

model.save("helmet_classifier.h5")

Found 661 images belonging to 2 classes.
Found 165 images belonging to 2 classes.
Epoch 1/5
42/42 [==============================] - 15s 325ms/step - loss: 0.2036 - accuracy: 0.9092 - val_loss: 0.0192 - val_accuracy: 1.0000
Epoch 2/5
42/42 [==============================] - 13s 310ms/step - loss: 0.0181 - accuracy: 1.0000 - val_loss: 0.0077 - val_accuracy: 1.0000
Epoch 3/5
42/42 [==============================] - 38s 916ms/step - loss: 0.0103 - accuracy: 1.0000 - val_loss: 0.0056 - val_accuracy: 1.0000
Epoch 4/5
42/42 [==============================] - 25s 583ms/step - loss: 0.0071 - accuracy: 1.0000 - val_loss: 0.0042 - val_accuracy: 1.0000
Epoch 5/5
42/42 [==============================] - 16s 387ms/step - loss: 0.0053 - accuracy: 1.0000 - val_loss: 0.0031 - val_accuracy: 1.0000


c:\Users\Siranjeevi\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [2]:
import cv2
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model("helmet_classifier.h5")

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = cv2.resize(frame, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)[0][0]

    if pred > 0.5:
        label = "NO HELMET"
        color = (0,0,255)
    else:
        label = "HELMET"
        color = (0,255,0)

    cv2.putText(frame, label, (50,50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    cv2.imshow("Helmet Classifier", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 79ms/step
